### Ejercicio Adicional

Solución del ejercicio + explicaciones sobre la lógica utilizada.

- `inicializar`: crea las carpetas `originales`, `procesadas` y `burlas`.
- `procesar`: valida la primera linea de cada entrega, extrae el padron y guarda el contenido sin la cabecera.
- `burlarme`: genera una version del texto reemplazando vocales por `i`.

Aclaraciones:
- Validacion de argumentos para evitar ejecutar el script sin accion o sin directorio.
- Se controla el caso en el que no hay archivos `.txt`, para que el `for` no intente procesar un patron literal.
- En `procesar` se elimina solo la primera linea del archivo validado.
- Mensajes claros para el usuario.

In [ ]:
#!/usr/bin/env bash

set -euo pipefail

ACCION=${1:-}
DIRECTORIO=${2:-}

uso() {
  echo "Uso: bash solucion.sh <inicializar|procesar|burlarme> <directorio>"
}

listar_txt() {
  local carpeta=$1
  find "$carpeta" -maxdepth 1 -type f -name '*.txt' | sort
}

if [ -z "$ACCION" ] || [ -z "$DIRECTORIO" ]; then
  uso
  exit 1
fi

inicializar() {
  local directorio_padre=$1

  for nombre in originales procesadas burlas; do
    local directorio="$directorio_padre/$nombre"

    if [ -d "$directorio" ]; then
      echo "El directorio $directorio ya existe"
    else
      mkdir -p "$directorio"
      echo "Se creo correctamente $directorio"
    fi
  done
}

procesar() {
  local directorio=$1
  local origen="$directorio/originales"
  local destino="$directorio/procesadas"
  local regex='^Alumno: [A-Za-z]+( [A-Za-z]+)*, [A-Za-z]+( [A-Za-z]+)* - Padron: [0-9]{6}$'
  local archivos=()

  if [ ! -d "$origen" ] || [ ! -d "$destino" ]; then
    echo "Faltan directorios necesarios. Ejecuta primero la accion inicializar."
    return 1
  fi

  mapfile -t archivos < <(listar_txt "$origen")

  if [ "${#archivos[@]}" -eq 0 ]; then
    echo "No hay archivos .txt para procesar en $origen"
    return 0
  fi

  for archivo in "${archivos[@]}"; do
    local primera_linea
    local nombre_archivo
    local padron

    primera_linea=$(head -n 1 "$archivo")
    nombre_archivo=$(basename "$archivo")

    if ! printf '%s\n' "$primera_linea" | grep -qE "$regex"; then
      echo "El archivo $nombre_archivo no cumple el enunciado."
      continue
    fi

    padron=$(printf '%s\n' "$primera_linea" | grep -oE '[0-9]{6}')
    tail -n +2 "$archivo" > "$destino/${padron}.txt"
    echo "Procesamos correctamente $nombre_archivo -> ${padron}.txt"
  done
}

burlarme() {
  local directorio=$1
  local origen="$directorio/procesadas"
  local destino="$directorio/burlas"
  local archivos=()

  if [ ! -d "$origen" ] || [ ! -d "$destino" ]; then
    echo "Faltan directorios necesarios. Ejecuta primero la accion inicializar y luego procesar."
    return 1
  fi

  mapfile -t archivos < <(listar_txt "$origen")

  if [ "${#archivos[@]}" -eq 0 ]; then
    echo "No hay archivos .txt para burlarse en $origen"
    return 0
  fi

  for archivo in "${archivos[@]}"; do
    local nombre

    nombre=$(basename "$archivo")
    sed 's/[aeou]/i/g; s/[AEOU]/I/g' "$archivo" > "$destino/$nombre"
    echo "Burla generada: $destino/$nombre"
  done
}

case "$ACCION" in
  inicializar) inicializar "$DIRECTORIO" ;;
  procesar) procesar "$DIRECTORIO" ;;
  burlarme) burlarme "$DIRECTORIO" ;;
  *)
    echo "Error: la accion $ACCION no es conocida"
    echo "Las validas son: inicializar, procesar, burlarme"
    uso
    exit 1
    ;;
esac

exit 0


## Lógica del script

Primero se guardan los argumentos en `ACCION` y `DIRECTORIO`. Eso permite reutilizarlos al final en el `case`, que decide que funcion ejecutar.  

La funcion `listar_txt` busca archivos `.txt`. De esta forma, `procesar` y `burlarme` no repiten logica y ademas quedan ordenados los archivos antes de recorrerlos.  

En `inicializar` se usa un `for` para evitar escribir tres veces la misma estructura. La idea es recorrer los nombres de las carpetas esperadas y crear solo las que faltan.  

En `procesar` se valida la primera linea con una expresion regular. Si la cabecera tiene el formato correcto, se extrae el padrón y se guarda un nuevo archivo en `procesadas` usando ese número como nombre. Además, permite nombres y apellidos compuestos separados por espacio, siempre que respeten el formato general pedido.  

La linea `tail -n +2` copia el archivo desde la segunda linea en adelante.  

En `burlarme` se usa `sed` para reemplazar vocales por `i`. El resultado se guarda en la carpeta `burlas` con el mismo nombre del archivo procesado.  